# GP Additive Kernel Signal Seperation

This notebook separates a target signal into two latent components with an additive Gaussian Process (GP) kernel.

We model the observed value as:
$$y(t, T) = f_{time}(t) + f_{temp}(T) + \epsilon$$
where one component depends on time and the other on temperature.

The final cell creates an animation with four stages:
1. Observed data only.
2. Recovered time component plus attribution bar.
3. Recovered temperature component plus attribution bars.
4. Full additive reconstruction overlay.

In [22]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from pathlib import Path
import seaborn as sns

from gp_utils import gp_posterior

sns.set_theme(style="whitegrid", palette="husl")
sns.set_context("notebook", font_scale=1.1)

In [23]:
rng = np.random.default_rng(12)

n_points = 140
time = np.linspace(0.0, 48.0, n_points)
temperature = rng.uniform(8.0, 34.0, size=n_points)


def time_signal_fn(t):
    # Smoother, slower time dynamics.
    return 1.85 * np.sin(2.0 * np.pi * t / 30.0) + 0.35 * np.cos(2.0 * np.pi * t / 60.0)


def temp_signal_fn(temp):
    centered = temp - 21.0
    # More fluctuative temperature response.
    return 0.55 * np.sin(1.2 * centered) + 0.35 * np.sin(2.4 * centered + 0.4) + 0.08 * centered


f_time = time_signal_fn(time)
f_temp = temp_signal_fn(temperature)

# Center both latent parts for a clean additive split.
f_time = f_time - np.mean(f_time)
f_temp = f_temp - np.mean(f_temp)

sn = 0.16
noise = rng.normal(0.0, sn, size=n_points)

y = f_time + f_temp + noise
X_train = np.column_stack([time, temperature])

print(f"Created dataset with {n_points} samples.")
print(f"time range: [{time.min():.1f}, {time.max():.1f}]")
print(f"temperature range: [{temperature.min():.1f}, {temperature.max():.1f}]")

Created dataset with 140 samples.
time range: [0.0, 48.0]
temperature range: [8.1, 33.0]


## Additive GP decomposition

We use a sum kernel:
$$k((t, T), (t', T')) = k_t(t, t') + k_T(T, T')$$

With this setup, each latent component posterior mean is:
$$\mu_t(X_*) = K_t(X_*, X)(K_t + K_T + \sigma_n^2 I)^{-1}y$$
$$\mu_T(X_*) = K_T(X_*, X)(K_t + K_T + \sigma_n^2 I)^{-1}y$$

In [24]:
def rbf_1d(x1, x2, lengthscale=1.0, variance=1.0):
    x1 = np.asarray(x1)
    x2 = np.asarray(x2)
    d2 = np.subtract.outer(x1, x2) ** 2
    return variance * np.exp(-0.5 * d2 / (lengthscale**2))


def kernel_time(X1, X2):
    return rbf_1d(X1[:, 0], X2[:, 0], lengthscale=7.5, variance=2.0)


def kernel_temp(X1, X2):
    return rbf_1d(X1[:, 1], X2[:, 1], lengthscale=0.95, variance=1.4)


def kernel_additive(X1, X2):
    return kernel_time(X1, X2) + kernel_temp(X1, X2)


K_time = kernel_time(X_train, X_train)
K_temp = kernel_temp(X_train, X_train)
K_total = K_time + K_temp

Ky = K_total + sn**2 * np.eye(len(X_train))
L = np.linalg.cholesky(Ky + 1e-10 * np.eye(len(X_train)))
alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))

mu_time_train = K_time @ alpha
mu_temp_train = K_temp @ alpha
mu_total_train = mu_time_train + mu_temp_train

mu_total_check, _ = gp_posterior(X_train, y, X_train, kernel_additive, sn)
print(f"max |manual - gp_utils|: {np.max(np.abs(mu_total_train - mu_total_check)):.2e}")

time_grid = np.linspace(time.min(), time.max(), 300)
temp_grid = np.linspace(temperature.min(), temperature.max(), 300)

X_time_grid = np.column_stack([time_grid, np.full_like(time_grid, np.mean(temperature))])
X_temp_grid = np.column_stack([np.full_like(temp_grid, np.mean(time)), temp_grid])

mu_time_grid = kernel_time(X_time_grid, X_train) @ alpha
mu_temp_grid = kernel_temp(X_temp_grid, X_train) @ alpha

time_center = np.mean(time_signal_fn(time))
temp_center = np.mean(temp_signal_fn(temperature))
true_time_grid = time_signal_fn(time_grid) - time_center
true_temp_grid = temp_signal_fn(temp_grid) - temp_center

component_var = np.array([np.var(mu_time_train), np.var(mu_temp_train)])
component_pct = 100.0 * component_var / np.sum(component_var)
print("Component attribution (variance share):")
print(f"  Time:        {component_pct[0]:5.1f}%")
print(f"  Temperature: {component_pct[1]:5.1f}%")

# Build two stacked parts that sum exactly to each observed point.
residual = y - mu_total_train
abs_component = np.abs(mu_time_train) + np.abs(mu_temp_train) + 1e-12
time_resid_share = np.abs(mu_time_train) / abs_component
temp_resid_share = 1.0 - time_resid_share

bar_time_part = mu_time_train + residual * time_resid_share
bar_temp_part = mu_temp_train + residual * temp_resid_share

sort_idx = np.argsort(time)
time_sorted = time[sort_idx]
y_sorted = y[sort_idx]
mu_total_sorted = mu_total_train[sort_idx]
true_total_sorted = (f_time + f_temp)[sort_idx]
bar_time_sorted = bar_time_part[sort_idx]
bar_temp_sorted = bar_temp_part[sort_idx]

print(f"max |(time part + temp part) - y|: {np.max(np.abs((bar_time_part + bar_temp_part) - y)):.2e}")

max |manual - gp_utils|: 4.22e-14
Component attribution (variance share):
  Time:         69.6%
  Temperature:  30.4%
max |(time part + temp part) - y|: 4.44e-16


In [25]:
frame_duration_ms = 1700
gif_fps = 1000.0 / frame_duration_ms

bar_sum_sorted = bar_time_sorted + bar_temp_sorted

main_ymin = min(np.min(y), np.min(mu_total_train), np.min(f_time + f_temp), np.min(bar_sum_sorted), np.min(bar_time_sorted))
main_ymax = max(np.max(y), np.max(mu_total_train), np.max(f_time + f_temp), np.max(bar_sum_sorted), np.max(bar_time_sorted))
main_margin = 0.12 * (main_ymax - main_ymin)
main_ylim = (main_ymin - main_margin, main_ymax + main_margin)

time_ymin = min(np.min(true_time_grid), np.min(mu_time_grid))
time_ymax = max(np.max(true_time_grid), np.max(mu_time_grid))
time_margin = 0.12 * (time_ymax - time_ymin)
time_ylim = (time_ymin - time_margin, time_ymax + time_margin)

temp_ymin = min(np.min(true_temp_grid), np.min(mu_temp_grid))
temp_ymax = max(np.max(true_temp_grid), np.max(mu_temp_grid))
temp_margin = 0.12 * (temp_ymax - temp_ymin)
temp_ylim = (temp_ymin - temp_margin, temp_ymax + temp_margin)

frame_titles = [
    "Frame 1/4: Observed data",
    "Frame 2/4: Time contribution bars on main plot",
    "Frame 3/4: Add temperature contribution bars",
    "Frame 4/4: Full additive reconstruction",
]

fig = plt.figure(figsize=(14, 9))
grid = fig.add_gridspec(2, 2, height_ratios=[1.65, 1.0])

ax_main = fig.add_subplot(grid[0, :])
ax_time = fig.add_subplot(grid[1, 0])
ax_temp = fig.add_subplot(grid[1, 1])

bar_width = (time.max() - time.min()) / len(time) * 0.88


def draw_frame(frame_idx):
    ax_main.cla()
    ax_time.cla()
    ax_temp.cla()

    shown_time = bar_time_sorted if frame_idx >= 1 else np.zeros_like(bar_time_sorted)
    shown_temp = bar_temp_sorted if frame_idx >= 2 else np.zeros_like(bar_temp_sorted)

    ax_main.bar(
        time_sorted,
        shown_time,
        width=bar_width,
        color="#1f77b4",
        alpha=0.35,
        edgecolor="none",
        label="Time contribution",
        zorder=1,
    )
    ax_main.bar(
        time_sorted,
        shown_temp,
        width=bar_width,
        bottom=shown_time,
        color="#2ca02c",
        alpha=0.35,
        edgecolor="none",
        label="Temperature contribution",
        zorder=1,
    )

    ax_main.scatter(time, y, s=24, color="steelblue", alpha=0.80, label="Observed data", zorder=3)

    if frame_idx >= 2:
        ax_main.plot(
            time_sorted,
            shown_time + shown_temp,
            color="#2f3640",
            lw=1.1,
            alpha=0.7,
            label="Stacked bar top",
            zorder=4,
        )

    if frame_idx >= 3:
        ax_main.plot(time_sorted, mu_total_sorted, color="#e67e22", lw=2.3, label="Recovered total", zorder=5)
        ax_main.plot(time_sorted, true_total_sorted, "--", color="#7f8c8d", lw=1.6, label="True total", zorder=5)

    ax_main.set_title(frame_titles[frame_idx], fontsize=14, fontweight="bold")
    ax_main.set_xlabel("Time")
    ax_main.set_ylabel("Target y")
    ax_main.set_xlim(time.min(), time.max())
    ax_main.set_ylim(*main_ylim)
    ax_main.grid(True, alpha=0.3)
    ax_main.legend(loc="upper right", ncols=2, fontsize=9, frameon=True, shadow=True)

    ax_time.set_title("Time signal overlay")
    ax_time.set_xlabel("Time")
    ax_time.set_ylabel("Component value")
    ax_time.set_xlim(time_grid.min(), time_grid.max())
    ax_time.set_ylim(*time_ylim)
    ax_time.grid(True, alpha=0.3)
    if frame_idx >= 1:
        ax_time.plot(time_grid, true_time_grid, "--", color="#7f8c8d", lw=1.4, label="True time signal")
        ax_time.plot(time_grid, mu_time_grid, color="#1f77b4", lw=2.3, label="Recovered time signal")
        ax_time.legend(loc="upper right", fontsize=9, frameon=True)
    else:
        ax_time.text(
            0.5,
            0.5,
            "Revealed in frame 2",
            transform=ax_time.transAxes,
            ha="center",
            va="center",
            fontsize=11,
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
        )

    ax_temp.set_title("Temperature signal overlay")
    ax_temp.set_xlabel("Temperature")
    ax_temp.set_ylabel("Component value")
    ax_temp.set_xlim(temp_grid.min(), temp_grid.max())
    ax_temp.set_ylim(*temp_ylim)
    ax_temp.grid(True, alpha=0.3)
    if frame_idx >= 2:
        ax_temp.plot(temp_grid, true_temp_grid, "--", color="#7f8c8d", lw=1.4, label="True temp signal")
        ax_temp.plot(temp_grid, mu_temp_grid, color="#2ca02c", lw=2.3, label="Recovered temp signal")
        ax_temp.legend(loc="upper right", fontsize=9, frameon=True)
    else:
        ax_temp.text(
            0.5,
            0.5,
            "Revealed in frame 3",
            transform=ax_temp.transAxes,
            ha="center",
            va="center",
            fontsize=11,
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
        )

    plt.tight_layout()
    return []


anim = animation.FuncAnimation(
    fig,
    draw_frame,
    frames=4,
    interval=frame_duration_ms,
    repeat=True,
)

save_path = Path("imgs") / "gp_signal_seperation_anim.gif"
save_path.parent.mkdir(parents=True, exist_ok=True)
anim.save(save_path, writer=animation.PillowWriter(fps=gif_fps))
print(f"Saved animation to {save_path}")

plt.close(fig)
HTML(anim.to_jshtml())

Saved animation to imgs\gp_signal_seperation_anim.gif


<Figure size 640x480 with 0 Axes>